# Transcriptor de audio gratuito

Este proyecto permite transcribir archivos de audio (`.m4a`, `.mp3`, etc.) a texto utilizando `faster-whisper`, optimizado para CPU en caso de no contar con GPU.

Es un proyecto exprés que desarrollé para no tener que pagar por aplicaciones externas que transcriban el audio de las múltiples entrevistas que tendré que llevar a cabo durante mi tesis.

Es una versión beta y con muuuucha oportunidad de mejora, empezando con que no logra diferenciar entre locutores. Estaré trabajando en eso cuando pueda y subiré avances ;)

## Paso opcional: Activar GPU en Colab (Recomendable)
Antes de ejecutar el código, ve al menú superior:
Entorno de ejecución > Cambiar tipo de entorno de ejecución > en Acelerador de hardware selecciona T4 GPU (o cualquier GPU disponible) y haz clic en Guardar. Esto acelerará drásticamente la velocidad del transcriptor.
### Sobre los modelos de GPU:
*   GPU T4 (Recomendada): Es la GPU estándar, consume la menor cantidad de unidades de cómputo en Colab (o es gratuita según la disponibilidad) y ofrece potencia más que suficiente para procesar audios rápidamente con Whisper.
*   GPU L4 o G4: Son GPUs más modernas y rápidas para modelos pesados o entrenamiento, pero para un transcriptor ligero consumirán tus créditos o cuota mucho más rápido sin una diferencia perceptible de velocidad.
*   GPU A100: Es excesiva para este propósito; está pensada para entrenar o ejecutar modelos masivos de lenguaje (LLMs).

## 1. Instalación de dependencias
Instala faster-whisper y el cliente de IA para la corrección de texto.

In [ ]:
# Instala la librería original de OpenAI Whisper y google-genai para la corrección
!pip install -q --upgrade openai-whisper google-genai

## 2. Cargar el modelo Whisper
Aprovecha la GPU si está disponible (con float16), o usa CPU (int8) automáticamente.

In [ ]:
import torch
import whisper

# Detectar automáticamente si hay GPU activa en Colab
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cargando OpenAI Whisper en: {device.upper()}")

# Cargamos el modelo base original
model = whisper.load_model("base", device=device)

Cargando OpenAI Whisper en: CUDA


## 3. Subir el archivo de audio
Permite seleccionar el audio directamente desde tu equipo o móvil ;)

In [ ]:
import os
from google.colab import files

print("Sube tu archivo de audio:")
uploaded = files.upload()

audio_path = list(uploaded.keys())[0]
print(f"Archivo listo: {audio_path}")

Sube tu archivo de audio:


## 4. Transcripción por pausas, corrección con IA y descarga
Nota sobre la IA gratuita: Si quieres activar la corrección ortográfica, obtén una clave sin costo en Google AI Studio y pégala en GEMINI_API_KEY. Si prefieres dejarla vacía (""), el script saltará la revisión y guardará la transcripción limpia directamente.
*   En mi caso cuento con Google AI gracias a la suscripción gratuita de 1 año que ofrecen para estudiantes ⚛

In [7]:
import textwrap
import time
import random
from google.colab import files

# ================= Configuración =================
# Si tienes una API Key gratuita de Google AI Studio, pégala aquí:
GEMINI_API_KEY = ""
# Ejemplo: "AIzaSy..."

# Pausa mínima (en segundos) de silencio entre frases para abrir un nuevo bloque
UMBRAL_SILENCIO_SEGUNDOS = 2.0

# Límite máximo de duración por bloque si no hay silencios (ej. 3 minutos = 180s)
MAX_DURACION_BLOQUE = 180
ANCHO_LINEA = 100

# Configuración de Reintentos Automáticos de la IA
MAX_REINTENTOS = 10
BACKOFF_BASE = 4.0
# =================================================

def formatear_tiempo(segundos):
    horas = int(segundos // 3600)
    minutos = int((segundos % 3600) // 60)
    segundos_restantes = int(segundos % 60)
    return f"{horas:02d}:{minutos:02d}:{segundos_restantes:02d}"

def corregir_con_gemini(texto, api_key):
    """Corrige errores gramaticales y ortográficos manteniendo el significado intacto."""
    from google import genai
    client = genai.Client(api_key=api_key)
    prompt = (
        "Eres un corrector de estilo profesional. Revisa la siguiente transcripción "
        "de audio en español. Corrige únicamente errores gramaticales, ortográficos, puntuación "
        "o palabras mal interpretadas por el transcriptor fonético. NO cambies el significado, "
        "no omitas contenido y devuelve exclusivamente el texto corregido sin introducciones ni comentarios.\n\n"
        f"Texto:\n{texto}"
    )
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    return response.text.strip()

print("Transcribiendo audio...")
# Transcribimos usando la librería original de whisper
result = model.transcribe(audio_path, language="es")
segmentos_lista = result.get("segments", [])

bloques = []

if segmentos_lista:
    inicio_bloque = segmentos_lista[0]["start"]
    ultimo_fin = segmentos_lista[0]["end"]
    texto_bloque = segmentos_lista[0]["text"].strip()

    for seg in segmentos_lista[1:]:
        silencio_previo = seg["start"] - ultimo_fin
        duracion_acumulada = seg["end"] - inicio_bloque

        # Corta si hay una pausa prolongada o si el bloque ya excedió el tiempo máximo
        if silencio_previo >= UMBRAL_SILENCIO_SEGUNDOS or duracion_acumulada > MAX_DURACION_BLOQUE:
            bloques.append((inicio_bloque, texto_bloque))
            inicio_bloque = seg["start"]
            texto_bloque = seg["text"].strip()
        else:
            texto_bloque += " " + seg["text"].strip()

        ultimo_fin = seg["end"]

    bloques.append((inicio_bloque, texto_bloque))

# Revisión con IA si hay clave configurada
bloques_finales = []
for i, (inicio, texto) in enumerate(bloques, 1):
    texto_limpio = " ".join(texto.split())
    if GEMINI_API_KEY:
        reintentos = 0
        while reintentos < MAX_REINTENTOS:
            print(f"Revisando con IA bloque {i}/{len(bloques)}... (Intento {reintentos + 1}/{MAX_REINTENTOS})")
            try:
                texto_limpio = corregir_con_gemini(texto_limpio, GEMINI_API_KEY)
                break # Éxito: salimos del bucle de reintento
            except Exception as e:
                reintentos += 1
                if reintentos >= MAX_REINTENTOS:
                    print(f"\n⚠️ Se alcanzó el límite máximo de reintentos para el bloque {i}. Se usará el texto original.")
                    break

                # Tiempo de espera exponencial con variación aleatoria para evitar colisiones
                tiempo_espera = (BACKOFF_BASE ** reintentos) + random.uniform(1.0, 3.0)
                print(f"\n⚠️ Falló la llamada a la IA ({e}).")
                print(f"Reintentando automáticamente en {tiempo_espera:.1f} segundos sin intervención...\n")
                time.sleep(tiempo_espera)

    bloques_finales.append((inicio, texto_limpio))

# Guardar y formatear archivo .txt
nombre_base = os.path.splitext(audio_path)[0]
archivo_salida = f"{nombre_base}_transcripcion.txt"

with open(archivo_salida, "w", encoding="utf-8") as f:
    for inicio, texto in bloques_finales:
        texto_formateado = textwrap.fill(
            texto,
            width=ANCHO_LINEA,
            break_long_words=False,
            break_on_hyphens=False
        )
        f.write(f"[{formatear_tiempo(inicio)}]\n")
        f.write(texto_formateado)
        f.write("\n\n")

print(f"\nProceso finalizado. Archivo guardado como: {archivo_salida}")
files.download(archivo_salida)

Transcribiendo audio...
Revisando con IA bloque 1/1... (Intento 1/10)

⚠️ Falló la llamada a la IA (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 2h58m21.251397745s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjec

KeyboardInterrupt: 